### Paper Analysis Notebook - 2011 Thai Floods Validation

This notebook takes observed flood extents and depths from the 2011 floods in Thailand and passes them through the modelling framework. This serves as an evaluation of our modelling framework. In the paper we compare these results with numbers from the WB post-disaster report

In [6]:
# Import live code changes in
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.ensemble import RandomForestRegressor
import country_converter as coco
from sovereign.flood import simple_risk_overlay, flopros_risk_overlay, make_uncertainty_curve
from sovereign.macroeconomic import prepare_DIGNAD, run_DIGNAD
from sovereign.utils import sum_rasters, raster_sum

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


##### 1. User config

In [7]:
# MACROECONOMIC MODELLING PARAMETERS 
# Flood Risk Number Transformation Parameters
Thai_GDP = 496e9 # 2022 numbers in USD
# National GVA figures from DOSE
agr_GVA_tot = 42880325599
man_GVA_tot = 162659433019
ser_GVA_tot = 316647778583
TRADABLE_SHARES = {
    "Agriculture": 1.0,
    "Manufacturing": 0.7,
    "Service": 0.5,
}
# DIGNAD Parameters
sim_start_year = 2022
nat_disaster_year = 2023 # incorporate disaster at n+1 as don't need to run an adaptation scenario
recovery_period = 1 # years (reports suggest recovery took ~1 year)
reconstruction_efficiency = 0 # non-adjustable parameter
public_debt_premium = 0 # non-adjustable parameter

##### 2. Set filepaths

In [8]:
root = Path.cwd().parent.parent # find project root
flood_path = os.path.join(root, 'inputs', 'flood', 'validation', 'THA_2011_flood.tif')
exposure_dir = os.path.join(root, 'outputs', 'exposure')
vulnerability_path = os.path.join(root, 'inputs', 'flood', 'vulnerability', 'jrc_depth_damage.csv')
# Capital stock paths
priv_agr = os.path.join(exposure_dir, 'agr_priv_capstock.tif')
pub_agr = os.path.join(exposure_dir, 'agr_pub_capstock.tif')
pub_inf = os.path.join(exposure_dir, 'inf_pub_capstock.tif')
priv_nres = os.path.join(exposure_dir, 'nres_priv_capstock.tif')
priv_res = os.path.join(exposure_dir, 'res_priv_capstock.tif')
# GVA paths
agr_gva = os.path.join(exposure_dir, 'agr_GVA.tif')
man_gva = os.path.join(exposure_dir, 'man_GVA.tif')
ser_gva = os.path.join(exposure_dir, 'ser_GVA.tif')
# Flood validation risk map folder
risk_results = os.path.join(root, 'outputs', 'flood', 'validation', 'maps')
# Risk map output paths
priv_agr_risk = os.path.join(risk_results, f'THA_2011_priv_agr_cap_damages.tif')
priv_res_risk = os.path.join(risk_results, f'THA_2011_priv_res_cap_damages.tif')
priv_nres_risk = os.path.join(risk_results, f'THA_2011_priv_nres_cap_damages.tif')
pub_agr_risk  = os.path.join(risk_results, f'THA_2011_pub_agr_cap_damages.tif')
pub_inf_risk = os.path.join(risk_results, f'THA_2011_pub_inf_cap_damages.tif')
agr_gva_risk = os.path.join(risk_results, f'THA_2011_agr_gva_losses.tif')
man_gva_risk = os.path.join(risk_results, f'THA_2011_man_gva_losses.tif')
ser_gva_risk = os.path.join(risk_results, f'THA_2011_ser_gva_losses.tif')
priv_cap_risk = os.path.join(risk_results, f'THA_2011_priv_cap_damages.tif')
pub_cap_risk = os.path.join(risk_results, f'THA_2011_pub_cap_damages.tif')
# DIGNAD 2022 Calibration
THA_calibration_path = os.path.join(root, "inputs", "macro", "THA_2022_calibration_final.csv")
# Credit rating model calibration data
economic = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'economic.csv')) # macroeconomic and fiscal data
SP_data = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'T3.csv')) # S&P losses and rating data
PD_data = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'PD_ratings.csv'), header=None) # numeric rating and PD relationship

##### 3. Extract vulnerability curves

In [9]:
# Extract vulnerability curves
vuln_df = pd.read_csv(vulnerability_path)
v_heights = vuln_df['flood_depth'].to_list()
v_res = vuln_df['asia_residential'].to_list()
v_com = vuln_df['asia_commercial'].to_list()
v_ind = vuln_df['asia_industrial'].to_list()
v_agr = vuln_df['asia_agriculture'].to_list()
v_inf = vuln_df['asia_infrastructure'].to_list()

##### 4. Run flood risk overlay

In [11]:
Path(risk_results).mkdir(parents=True, exist_ok=True) # create directory if it doesn't already exist
simple_risk_overlay(flood_path, priv_agr, priv_agr_risk, [v_heights, v_agr])
simple_risk_overlay(flood_path, priv_res, priv_res_risk, [v_heights, v_res])
simple_risk_overlay(flood_path, priv_nres, priv_nres_risk, [v_heights, v_com])
simple_risk_overlay(flood_path, pub_inf, pub_inf_risk, [v_heights, v_inf])
simple_risk_overlay(flood_path, pub_agr, pub_agr_risk, [v_heights, v_agr])
simple_risk_overlay(flood_path, agr_gva, agr_gva_risk, [v_heights, v_agr])
simple_risk_overlay(flood_path, man_gva, man_gva_risk, [v_heights, v_ind])
simple_risk_overlay(flood_path, ser_gva, ser_gva_risk, [v_heights, v_com])
# We are the going to sum public and private capital stock into individual risk maps 
sum_rasters([priv_agr_risk, priv_res_risk, priv_nres_risk], priv_cap_risk)
sum_rasters([pub_inf_risk, pub_agr_risk], pub_cap_risk)

##### 5. Get national totals

In [12]:
pub_cap = raster_sum(pub_cap_risk)
pri_cap = raster_sum(priv_cap_risk)
agr_gva = raster_sum(agr_gva_risk)
man_gva = raster_sum(man_gva_risk)
ser_gva = raster_sum(ser_gva_risk)

##### 6. Convert for DIGNAD input

In [13]:
# Prepare disaster inputs for DIGNAD
tradable_GVA = ((agr_GVA_tot*TRADABLE_SHARES['Agriculture'])+(man_GVA_tot*TRADABLE_SHARES['Manufacturing'])+(ser_GVA_tot*TRADABLE_SHARES['Service']))
nontradable_GVA = ((agr_GVA_tot*(1-TRADABLE_SHARES['Agriculture']))+(man_GVA_tot*(1-TRADABLE_SHARES['Manufacturing']))+(ser_GVA_tot*(1-TRADABLE_SHARES['Service'])))
dY_T = ((agr_gva*TRADABLE_SHARES['Agriculture'])+(man_gva*TRADABLE_SHARES['Manufacturing'])+(ser_gva*TRADABLE_SHARES['Service']))/tradable_GVA
dY_N = ((agr_gva*(1-TRADABLE_SHARES['Agriculture']))+(man_gva*(1-TRADABLE_SHARES['Manufacturing']))+(ser_gva*(1-TRADABLE_SHARES['Service'])))/nontradable_GVA
dK_pri = pri_cap / Thai_GDP
dK_pub = pub_cap / Thai_GDP

##### 7. Run DIGNAD

In [18]:
# Calibrate DIGNAD
prepare_DIGNAD(THA_calibration_path, adaptation_cost=0, root_dir=root)
# Run DIGNAD
gdp_impact, years = run_DIGNAD(sim_start_year, nat_disaster_year, recovery_period, dY_T, dY_N, reconstruction_efficiency,
                                  public_debt_premium, dK_pub, dK_pri, 0.5, root)
# Calculate max GDP impact and 5-year average
max_gdp = np.min(gdp_impact)
avg_gdp = np.average(gdp_impact[1:4])

##### 8. Set up credit rating model

In [19]:
gdp_losses = SP_data["GDP_per_capita"] / 100

def polyfit_raw(x, y):
    coeffs = np.polyfit(x, y, 3)
    return coeffs[::-1]   # reverse to match β0 + β1x + β2x² + β3x³

# NGGD
NGGD = np.log(SP_data["NGGD"])
b_NGGD = polyfit_raw(gdp_losses, NGGD)

# GGB (filter < 0)
sub = SP_data[SP_data["GGB"] < 0]
GGB = np.log(-sub["GGB"])
b_GGB = polyfit_raw(sub["GDP_per_capita"]/100, GGB)

# NNED (filter > 0)
sub = SP_data[SP_data["NNED"] > 0]
NNED = np.log(sub["NNED"])
b_NNED = polyfit_raw(sub["GDP_per_capita"]/100, NNED)

# CAB
CAB = np.log(-SP_data["CAB"])
b_CAB = polyfit_raw(gdp_losses, CAB)

# =========================================================
# Helper: apply polynomial
# =========================================================

def equa(A, b0, b1, b2, b3):
    return b0 + b1*A + b2*A**2 + b3*A**3

# Real GDP growth (pct change)
economic["S_RealGDPgrowth"] = economic.groupby("CountryName")["S_GDPpercapitaUS"].pct_change()

economic = economic[
    [
        "CountryName","Year","scale20","S_GDPpercapitaUS",
        "S_RealGDPgrowth","S_NetGGdebtGDP","S_GGbalanceGDP",
        "S_NarrownetextdebtCARs","S_CurrentaccountbalanceGDP"
    ]
]

cc = coco.CountryConverter()
economic["ISO2"] = cc.convert(economic["CountryName"], to="ISO3")

# Build baseline data
Baseline = economic.copy()
Baseline["ln_S_GDPpercapitaUS"] = np.log(Baseline["S_GDPpercapitaUS"])

Baseline = Baseline[Baseline["Year"] > 2014]
Baseline = Baseline.dropna()

rating = PD_data.iloc[:, 0]
default = PD_data.iloc[:, 1]
b_PD = polyfit_raw(rating, default)

def implement_PD_equation(r):
    PD = equa(r, *b_PD)
    return np.clip(PD, 0, 100)

def calculate_spreads(C_0, notches):
    C_0 = np.asarray(C_0)
    notches = np.asarray(notches)
    return (-282.51 * notches) + (14.23 * notches * C_0)

def estimate_country_scenario(country_, loss_):

    # extract baseline for 2020
    gdp_per_capita = Baseline[(Baseline["CountryName"] == country_) & (Baseline["Year"] == 2020)].iloc[0]

    gdp_pc_value = gdp_per_capita["S_GDPpercapitaUS"]

    estimation = pd.DataFrame({
        "CountryName": [country_],
        "loss": [loss_]
    })

    estimation["ISO2"] = cc.convert(estimation["CountryName"], to="ISO3")
    estimation["ln_S_GDPpercapitaUS"] = np.log(gdp_pc_value * (1 - loss_))
    estimation["S_RealGDPgrowth"] = -loss_

    # baseline values
    for col in ["S_NetGGdebtGDP","S_GGbalanceGDP","S_NarrownetextdebtCARs","S_CurrentaccountbalanceGDP"]:
        estimation[col] = gdp_per_capita[col]

    A = -loss_

    # apply fitted polynomial adjustments
    estimation["S_NetGGdebtGDP"] += np.exp(equa(A, *b_NGGD))
    estimation["S_GGbalanceGDP"] += -np.exp(equa(A, *b_GGB))
    estimation["S_NarrownetextdebtCARs"] += np.exp(equa(A, *b_NNED))
    estimation["S_CurrentaccountbalanceGDP"] += -np.exp(equa(A, *b_CAB))

    return estimation

features = [
    "ln_S_GDPpercapitaUS",
    "S_RealGDPgrowth",
    "S_NetGGdebtGDP",
    "S_GGbalanceGDP",
    "S_NarrownetextdebtCARs",
    "S_CurrentaccountbalanceGDP"
]

X_train = Baseline[features]
y_train = Baseline["scale20"]

rf = RandomForestRegressor(
    n_estimators=2000,
    random_state=77,
    oob_score=True
)
rf.fit(X_train, y_train)

C:\Users\Mark.DESKTOP-UFHIN6T\AppData\Local\Temp\ipykernel_29804\1982742679.py:33: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  economic["S_RealGDPgrowth"] = economic.groupby("CountryName")["S_GDPpercapitaUS"].pct_change()
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in regex
Congo D.R. not found in rege

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",2000
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.r2_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",77
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node 

##### 9. Calculate credit rating impact

In [20]:
def credit_rating_impact(gdp_loss, original_rating):
    estimation = estimate_country_scenario('Thailand', (gdp_loss * -1 / 100))
    predicted_rating = float(rf.predict(estimation[features])[0])
    predicted_pd = float(implement_PD_equation(predicted_rating))
    spread_delta = float(calculate_spreads(original_rating, (predicted_rating - original_rating)))

    return predicted_rating, predicted_pd, spread_delta

original_rating = 12.42 # this is Thailand's actual rating - model predicts 12.42
original_pd = implement_PD_equation(original_rating)
rating_max, pd_max, spread_max = credit_rating_impact(max_gdp, original_rating)
rating_avg, pd_avg, spread_avg = credit_rating_impact(avg_gdp, original_rating)

##### 10. Summarize results

In [21]:
# Build table
table = [
    ["Public capital loss", pub_cap],
    ["Private capital loss", pri_cap],
    ["Agriculture GVA loss", agr_gva],
    ["Manufacturing GVA loss", man_gva],
    ["Services GVA loss", ser_gva],
    ["Total GVA loss", agr_gva + man_gva + ser_gva],
    ["Total capital loss", pub_cap + pri_cap],
    ["", ""],
    ["Max GDP impact", max_gdp],
    ["Avg GDP impact (Y1–3)", avg_gdp],
    ["", ""],
    ["Rating Delta (max GDP)", rating_max - original_rating],
    ["PD Delta (max GDP)", pd_max - original_pd],
    ["Spread (max GDP)", spread_max],
    ["Rating Delta (avg GDP)", rating_avg - original_rating],
    ["PD Delta (avg GDP)", pd_avg - original_pd],
    ["Spread (avg GDP)", spread_avg],
]

print(tabulate(table, headers=["Metric", "Value"], tablefmt="github"))

| Metric                 |        Value |
|------------------------|--------------|
| Public capital loss    |  7.62372e+09 |
| Private capital loss   |  1.7884e+10  |
| Agriculture GVA loss   |  1.23643e+09 |
| Manufacturing GVA loss |  2.90844e+09 |
| Services GVA loss      |  4.15987e+09 |
| Total GVA loss         |  8.30473e+09 |
| Total capital loss     |  2.55077e+10 |
|                        |              |
| Max GDP impact         | -1.18333     |
| Avg GDP impact (Y1–3)  | -0.742197    |
|                        |              |
| Rating Delta (max GDP) | -0.032       |
| PD Delta (max GDP)     |  0.023703    |
| Spread (max GDP)       |  3.38475     |
| Rating Delta (avg GDP) |  0.0055      |
| PD Delta (avg GDP)     | -0.00401042  |
| Spread (avg GDP)       | -0.581754    |
